# Etapa 2 - Pré-processamento da base NSL-KDD

**Entrega de 07/11 - Grupo 5**

Este notebook implementa e compara as decisões obrigatórias de pré-processamento. O classificador de referência serve apenas para medir o efeito das estratégias; a comparação definitiva de classificadores pertence à Etapa 3.

## 1. Decisões herdadas da Etapa 1

A análise exploratória encontrou 69.833 registros, 50 colunas, 94,16% de registros normais e 5,84% de anomalias. As regras abaixo são aplicadas de forma auditável:

- remoção de duplicatas exatas, preservando a primeira ocorrência;
- remoção de `num_outbound_cmds`, por ser constante;
- remoção de `src_bytes_kb`, `dst_bytes_kb` e `total_bytes`, mantendo as representações originais em bytes;
- exclusão preventiva de `tipo_alerta_soc`, `score_ids_legado` e `tempo_resposta_analista_min`, devido ao risco de informação operacional ou posterior ao evento;
- normalização de caixa e espaços em `protocol_type`, `service` e `flag`;
- valores impossíveis em taxas, contagens e grandezas não negativas são convertidos em ausentes, sem inventar fatores de escala;
- grupos com preditores idênticos permanecem na mesma dobra de validação.

Medianas, modas, categorias e parâmetros de escala são aprendidos somente no treino de cada dobra.

In [1]:
from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing_stage2 import (
    evaluate_cleaning_strategies,
    evaluate_preprocessing_strategies,
    prepare_modeling_data,
    report_as_dict,
)

pd.set_option('display.max_columns', 100)

## 2. Carregamento

O código procura primeiro a base usada na Etapa 1 e depois o arquivo oficial de treino. O conjunto de teste sem rótulo não é usado nos experimentos.

In [2]:
data_candidates = [
    PROJECT_ROOT / 'base_nsl_kdd.csv',
    PROJECT_ROOT / 'data' / 'base_nsl_kdd.csv',
    PROJECT_ROOT / 'data' / 'treino.csv',
]
DATA_PATH = next((path for path in data_candidates if path.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        'Base não encontrada. Adicione base_nsl_kdd.csv na raiz ou treino.csv em data/.'
    )

raw_data = pd.read_csv(DATA_PATH, sep=None, engine='python')
print('Arquivo:', DATA_PATH)
print('Dimensões brutas:', raw_data.shape)
display(raw_data.head())

Arquivo: /Users/samuelalves/Desktop/TPIA/data/treino.csv
Dimensões brutas: (69833, 50)


,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,num_failed_logins,logged_in,num_compromised,root_shell,su_attempted,num_root,num_file_creations,num_shells,num_access_files,num_outbound_cmds,is_host_login,is_guest_login,count,srv_count,serror_rate,srv_serror_rate,rerror_rate,srv_rerror_rate,same_srv_rate,diff_srv_rate,srv_diff_host_rate,dst_host_count,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,src_bytes_kb,dst_bytes_kb,total_bytes,same_srv_pct,protocolo_cod,score_ids_legado,tipo_alerta_soc,tempo_resposta_analista_min,class
0,0,tcp,http,SF,219.0,3729,0,0,0,0,0,1.0,0.0,0.0,0.0,0,NaN,0,0.0,0,0,0,3.0,4.0,NaN,0.0,0.0,0.0,1.0,0.0,0.50,255.0,255.0,1.00,NaN,0.00,0.00,0.0,0.0,0.0,0.0,0.213867,3.641602,3948.0,100.0,0,0.176,OK,NaN,normal
1,0,tcp,finger,SF,NaN,443,0,0,0,0,0,0.0,0.0,0.0,0.0,0,0.0,0,0.0,0,0,0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.00,223.0,NaN,0.04,0.03,0.00,0.00,0.0,0.0,0.0,0.0,NaN,0.432617,NaN,100.0,0,0.067,OK,NaN,anomaly
2,0,tcp,http,SF,162.0,29122,0,0,0,0,0,1.0,0.0,0.0,0.0,0,0.0,0,0.0,0,0,0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.00,255.0,255.0,1.00,0.00,0.00,0.00,NaN,0.0,0.0,0.0,0.158203,28.439453,29284.0,100.0,0,0.245,OK,NaN,normal
3,0,tcp,smtp,SF,942.0,332,0,0,0,0,0,1.0,0.0,0.0,0.0,0,0.0,0,0.0,0,0,0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.00,62.0,173.0,0.42,0.08,0.02,0.02,0.0,0.0,0.0,0.0,0.919922,0.324219,1274.0,100.0,0,0.239,OK,NaN,normal
4,0,tcp,http,SF,232.0,479,0,0,0,0,0,1.0,NaN,0.0,0.0,0,0.0,0,0.0,0,0,0,8.0,9.0,0.0,0.0,0.0,0.0,1.0,0.0,0.22,255.0,255.0,1.00,0.00,0.00,0.00,NaN,0.0,0.0,0.0,0.226562,0.467773,711.0,100.0,0,0.231,OK,NaN,normal


## 3. Comparação de redundâncias e inconsistências

As variantes abaixo alteram uma decisão de limpeza por vez: normalização categórica e remoção de redundâncias, recuperação determinística de bytes ausentes, valores inválidos mantidos versus convertidos em ausentes versus limitados, remoção de duplicatas e tratamento conservador dos conflitos. A limpeza de linhas ocorre apenas no treino de cada dobra.

In [3]:
cleaning_results = evaluate_cleaning_strategies(raw_data, n_splits=5)
display(cleaning_results.style.format({column: '{:.4f}' for column in cleaning_results.columns if column.endswith(('_media', '_desvio'))}))

,variante,f1_macro_media,f1_macro_desvio,precision_anomaly_media,recall_anomaly_media,tempo_ajuste_s_media
0,C5 + conflitos removidos,0.8745,0.0025,0.7449,0.7843,1.2177
1,C5b + conflitos por voto majoritário,0.8377,0.0045,0.6221,0.7928,2.1868
2,C0 sem tratamento,0.8370,0.0041,0.6180,0.7965,3.2429
3,C3 + inválidos -> NaN + imputação,0.8368,0.0027,0.6201,0.7919,2.1299
4,C1 + categorias normalizadas e redundantes removidos,0.8362,0.0037,0.6162,0.7958,1.8919
5,C2 + bytes recuperados pelas relações exatas,0.8362,0.0037,0.6162,0.7958,1.9004
6,C3b + inválidos -> clip na faixa,0.8357,0.0073,0.6142,0.7970,1.9554
7,C4 + duplicatas exatas removidas,0.8265,0.0017,0.5900,0.7941,1.7871


## 4. Tratamento determinístico escolhido

Essas regras foram definidas antes da validação e não estimam parâmetros a partir das dobras. Valores inválidos viram `NaN`; a imputação propriamente dita continua dentro do pipeline. Rótulos conflitantes não são alterados automaticamente.

In [4]:
X, y, preparation_report = prepare_modeling_data(raw_data)
display(pd.Series(report_as_dict(preparation_report), name='resultado').to_frame())
display(y.value_counts().rename('quantidade').to_frame())
print('Dimensões após regras determinísticas:', X.shape)

,resultado
target_column,class
original_rows,69833
exact_duplicates_removed,11355
final_rows,58478
dropped_leakage_columns,"(tipo_alerta_soc, score_ids_legado, tempo_resp..."
dropped_constant_columns,"(num_outbound_cmds,)"
dropped_redundant_columns,"(src_bytes_kb, dst_bytes_kb, total_bytes)"
invalid_values_replaced_by_nan,4887
categorical_columns,"(protocol_type, service, flag)"
numeric_columns,"(duration, src_bytes, dst_bytes, land, wrong_f..."


,quantidade
normal,54959
anomaly,3519


Dimensões após regras determinísticas: (58478, 42)


## 5. Técnicas comparadas

### Codificação simbólico-numérica

Utilizamos `OneHotEncoder(handle_unknown='ignore')`. A codificação não impõe ordem artificial e aceita categorias não vistas nas dobras de validação. Em `service`, que possui alta cardinalidade, o aumento de dimensão é controlado pela redução posterior.

### Imputação e transformação numérica

Atributos categóricos usam a moda. Atributos numéricos usam mediana e indicadores de ausência. A mediana é menos sensível a caudas e valores extremos. Depois, `StandardScaler` coloca os atributos numéricos em escala comparável.

### Redução de dimensionalidade

`TruncatedSVD(n_components=20)` reduz a matriz esparsa produzida pelo one-hot sem exigir centralização densa. Os mesmos 20 componentes são usados em todas as estratégias.

### Balanceamento

Comparamos `RandomUnderSampler` e `RandomOverSampler` contra um baseline sem balanceamento. O under-sampling reduz a classe majoritária e pode perder informação. O over-sampling replica exemplos minoritários e pode aumentar sobreajuste. Não usamos SMOTE nesta etapa porque interpolar diretamente categorias one-hot pode criar combinações categóricas fracionárias sem interpretação.

## 6. Protocolo experimental

A comparação usa cinco dobras de `StratifiedGroupKFold`, sem embaralhamento. A estratificação preserva aproximadamente a proporção das classes, enquanto o agrupamento impede que assinaturas idênticas apareçam simultaneamente no treino e na validação. Todo pré-processamento aprendido e todo balanceamento ocorrem dentro da dobra de treino.

Métricas: F1-macro, precisão e recall macro, precisão da anomalia e recall da anomalia. A acurácia não orienta a escolha devido ao desbalanceamento.

In [5]:
results = evaluate_preprocessing_strategies(
    X,
    y,
    preparation_report,
    n_splits=5,
    n_components=20,
)
display(results.style.format({column: '{:.4f}' for column in results.columns if column.endswith(('_media', '_desvio'))}))

,estrategia,codificacao,transformacao_numerica,reducao_dimensionalidade,classificador_referencia,dobras,f1_macro_media,f1_macro_desvio,precision_macro_media,precision_macro_desvio,recall_macro_media,recall_macro_desvio,precision_anomaly_media,precision_anomaly_desvio,recall_anomaly_media,recall_anomaly_desvio,tempo_ajuste_s_media
0,sem_balanceamento,one_hot,standard_scaler,truncated_svd_20,regressao_logistica,5,0.7814,0.0075,0.8494,0.0054,0.7379,0.0087,0.7310,0.0103,0.4874,0.0175,0.6710
1,random_over_sampler,one_hot,standard_scaler,truncated_svd_20,regressao_logistica,5,0.6844,0.0061,0.6445,0.0050,0.8111,0.0054,0.3082,0.0099,0.7269,0.0123,0.8711
2,random_under_sampler,one_hot,standard_scaler,truncated_svd_20,regressao_logistica,5,0.6820,0.0075,0.6425,0.0060,0.8116,0.0046,0.3040,0.0123,0.7306,0.0132,0.6651


## 7. Exportação dos resultados

A tabela abaixo é gerada pelo experimento, evitando números copiados manualmente. Ela pode ser usada diretamente no texto da entrega.

In [6]:
OUTPUT_DIR = PROJECT_ROOT / 'output' / 'etapa_2'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
cleaning_results.to_csv(OUTPUT_DIR / 'resultados_limpeza.csv', index=False)
results.to_csv(OUTPUT_DIR / 'resultados_preprocessamento.csv', index=False)
(OUTPUT_DIR / 'diagnostico_preparacao.json').write_text(
    json.dumps(report_as_dict(preparation_report), ensure_ascii=False, indent=2),
    encoding='utf-8',
)
print('Arquivos gerados em:', OUTPUT_DIR.resolve())

Arquivos gerados em: /Users/samuelalves/Desktop/TPIA/output/etapa_2


## 8. Interpretação orientada pelos resultados

A estratégia indicada para a próxima etapa é a de maior F1-macro médio, desde que o desvio entre dobras e o recall da classe `anomaly` não revelem instabilidade. Se o ganho de recall vier acompanhado de grande perda de precisão, a equipe deve explicitar esse custo. O resultado não define ainda o melhor classificador; apenas seleciona uma configuração de pré-processamento para a Etapa 3.

## Declaração de uso de IA generativa

Foi utilizada uma ferramenta de IA generativa como apoio à estruturação do pipeline, à redação das justificativas e à depuração do código. As decisões metodológicas, a execução e a validação dos resultados permanecem sob responsabilidade da equipe.